<a href="https://colab.research.google.com/github/mtofighi/ChilwaBasin/blob/main/Generate_Collections_AnyLogic_09032025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👋 Chilwa Basin Collections Generator for AnyLogic

Last updated: 03 Sep 2025

This notebook generates four collections from the 'Categorized' sheet of the Chilwa Basin dataset for AnyLogic simulations: col_ColumnNumbersFromExcel, col_VariableNamesFromExcel, col_ColumnNumbersFromEstimates, and col_VariableNamesFromEstimates. Variables are listed up to four per line, grouped by category with comments (including the first category). Outputs are saved to `/content/Malawi/ChilwaCollections2025/{date_str}` and Google Drive equivalent.

**Dataset**: ChilwaBasin_Dataset_08202025.xlsx, 'Categorized' sheet.
**Objective**: Generate AnyLogic-compatible collections with filtering based on the 'Use' column ('F' or 'T' for Excel, 'T' for Estimates).


# 🚧 Installation

Install required packages for data processing and file handling.


In [1]:
!python -m pip install --upgrade pip -q
!python -m pip install pandas openpyxl -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.6 MB/s eta 0:00:00


# 📚 Import Libraries

Import libraries for data processing and file operations.


In [6]:
import pandas as pd
import os
import shutil
from google.colab import drive
from datetime import datetime


# 📊 Generate Collections

Load the 'Categorized' sheet, filter based on the 'Use' column, and generate collections with up to four variables per line, grouped by category with comments (including the first category). Save to text files in local and Google Drive directories.


In [11]:
# Mount Google Drive
drive.mount('/content/drive')

# Define output directories
date_str = datetime.now().strftime('%Y%m%d')
output_dir = f'/content/Malawi/ChilwaCollections2025/{date_str}'
drive_output_dir = f'/content/drive/My Drive/Malawi/ChilwaCollections2025/{date_str}'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_output_dir, exist_ok=True)

# File paths
col_numbers_excel_file = f'{output_dir}/col_ColumnNumbersFromExcel.txt'
col_names_excel_file = f'{output_dir}/col_VariableNamesFromExcel.txt'
col_numbers_estimates_file = f'{output_dir}/col_ColumnNumbersFromEstimates.txt'
col_names_estimates_file = f'{output_dir}/col_VariableNamesFromEstimates.txt'
drive_col_numbers_excel_file = f'{drive_output_dir}/col_ColumnNumbersFromExcel.txt'
drive_col_names_excel_file = f'{drive_output_dir}/col_VariableNamesFromExcel.txt'
drive_col_numbers_estimates_file = f'{drive_output_dir}/col_ColumnNumbersFromEstimates.txt'
drive_col_names_estimates_file = f'{drive_output_dir}/col_VariableNamesFromEstimates.txt'

# Load the 'Categorized' sheet
url = 'https://github.com/mtofighi/ChilwaBasin/blob/main/ChilwaBasin_DataAnalysis_032024/Dataset/ChilwaBasin_Dataset_08202025.xlsx?raw=true'
try:
    categorized = pd.read_excel(url, sheet_name='Categorized')
except Exception as e:
    print(f"Error loading 'Categorized' sheet: {e}")
    raise

# Verify required columns
required_columns = ['Column_Number_in_the_Excel_Dataset', 'Variable_Name_in_AnyLogic', 'Category', 'Use']
if not all(col in categorized.columns for col in required_columns):
    missing_cols = [col for col in required_columns if col not in categorized.columns]
    raise ValueError(f"Missing columns in 'Categorized' sheet: {missing_cols}")

# Print headers to confirm loading
headers = categorized.columns.tolist()
print("Headers:", headers)

# Function to format collections with up to four items per line, including first category comment, no trailing comma
def format_collection(items, category):
    if not items:
        return f'// {category}\n'
    output = f'// {category}\n'
    for i in range(0, len(items), 4):
        line_items = items[i:i+4]
        output += ', '.join(str(item) for item in line_items) + '\n'
    return output

# Filter and group data
excel_data = categorized[categorized['Use'].isin(['F', 'T'])]
estimates_data = categorized[categorized['Use'] == 'T']

# Group by category
excel_groups = excel_data.groupby('Category')
estimates_groups = estimates_data.groupby('Category')

# Generate collections
col_numbers_excel = []
col_names_excel = []
col_numbers_estimates = []
col_names_estimates = []

for category, group in excel_groups:
    numbers = group['Column_Number_in_the_Excel_Dataset'].astype(str).tolist()
    names = group['Variable_Name_in_AnyLogic'].tolist()
    col_numbers_excel.append(format_collection(numbers, category))
    col_names_excel.append(format_collection(names, category))

for category, group in estimates_groups:
    numbers = group['Column_Number_in_the_Excel_Dataset'].astype(str).tolist()
    names = group['Variable_Name_in_AnyLogic'].tolist()
    col_numbers_estimates.append(format_collection(numbers, category))
    col_names_estimates.append(format_collection(names, category))

# Save collections to text files
try:
    with open(col_numbers_excel_file, 'w') as f:
        f.write(''.join(col_numbers_excel))
    shutil.copy(col_numbers_excel_file, drive_col_numbers_excel_file)
    print(f"Column numbers (Excel) saved to '{col_numbers_excel_file}' and copied to Google Drive.")
except Exception as e:
    print(f"Error saving col_ColumnNumbersFromExcel: {e}")

try:
    with open(col_names_excel_file, 'w') as f:
        f.write(''.join(col_names_excel))
    shutil.copy(col_names_excel_file, drive_col_names_excel_file)
    print(f"Variable names (Excel) saved to '{col_names_excel_file}' and copied to Google Drive.")
except Exception as e:
    print(f"Error saving col_VariableNamesFromExcel: {e}")

try:
    with open(col_numbers_estimates_file, 'w') as f:
        f.write(''.join(col_numbers_estimates))
    shutil.copy(col_numbers_estimates_file, drive_col_numbers_estimates_file)
    print(f"Column numbers (Estimates) saved to '{col_numbers_estimates_file}' and copied to Google Drive.")
except Exception as e:
    print(f"Error saving col_ColumnNumbersFromEstimates: {e}")

try:
    with open(col_names_estimates_file, 'w') as f:
        f.write(''.join(col_names_estimates))
    shutil.copy(col_names_estimates_file, drive_col_names_estimates_file)
    print(f"Variable names (Estimates) saved to '{col_names_estimates_file}' and copied to Google Drive.")
except Exception as e:
    print(f"Error saving col_VariableNamesFromEstimates: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Headers: ['Variable_Name_in_AnyLogic', 'Column_Header_in_the_Excel_Dataset', 'Column_Number_in_the_Excel_Dataset', 'Category', 'Use']
Column numbers (Excel) saved to '/content/Malawi/ChilwaCollections2025/20250903/col_ColumnNumbersFromExcel.txt' and copied to Google Drive.
Variable names (Excel) saved to '/content/Malawi/ChilwaCollections2025/20250903/col_VariableNamesFromExcel.txt' and copied to Google Drive.
Column numbers (Estimates) saved to '/content/Malawi/ChilwaCollections2025/20250903/col_ColumnNumbersFromEstimates.txt' and copied to Google Drive.
Variable names (Estimates) saved to '/content/Malawi/ChilwaCollections2025/20250903/col_VariableNamesFromEstimates.txt' and copied to Google Drive.
